In [ ]:
"""
SQUID vs. Simulation Overlay — Two-Panel Hysteresis Comparison
===============================================================
Loads the averaged SQUID CSV produced by `SQUID_full-analysis_Caps` and one
or more CONVERTED simulation CSVs, then overlays them on a two-panel figure:

  Top panel    : Raw physical moment — SQUID [emu] vs. simulation [A·m²]
                 Shows the data as measured / simulated without normalisation.
  Bottom panel : Normalised M/Msat — SQUID (M_norm) vs. simulation (Mz/Ms)
                 Allows direct shape comparison independent of sample volume.

SQUID averaged CSV columns expected (from SQUID_full-analysis_Caps):
  Field_T, M_norm_desc, M_norm_asc, M_emu_desc, M_emu_asc

Simulation CONVERTED CSV columns expected:
  B_ext (T), Mz/Ms, Moment_Am2

Workflow (GUI-driven):
  1. Select the averaged SQUID CSV.
  2. Select one or more simulation CSVs (one per diameter).

Inputs  : Averaged SQUID CSV + one or more CONVERTED simulation CSVs.
Outputs : PNG + SVG two-panel comparison plot, saved to a timestamped folder.
"""

# --- Standard library ---
import os
from datetime import datetime

# --- Third-party ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- GUI ---
import tkinter as tk
from tkinter import filedialog


# ===========================================================================
# 1. SETTINGS
# ===========================================================================

# Unit conversion: 1 emu = 1e-3 A·m²
EMU_TO_AM2 = 1e-3

# Keywords for auto-detecting the field column in simulation CSVs
SIM_FIELD_KEYWORDS = ['B_ext (T)', 'B_ext', 'Field', 'H_ext']

# The CONVERTED simulation CSVs have two magnetisation columns:
#   Mz/Ms      — normalised M/Msat → used in the bottom panel
#   Moment_Am2 — physical moment [A·m²], volume-corrected per diameter → top panel
SIM_MAG_NORM   = 'Mz/Ms'
SIM_MAG_PHYS   = 'Moment_Am2'


# ===========================================================================
# 2. MAIN
# ===========================================================================

def main():

    # --- File selection via GUI dialogs ---
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    print("Step 1: Select the averaged SQUID CSV (output of SQUID_full-analysis_Caps)...")
    squid_path = filedialog.askopenfilename(
        title="Select averaged SQUID CSV",
        filetypes=[("CSV files", "*.csv")]
    )

    print("Step 2: Select one or more CONVERTED simulation CSVs...")
    sim_paths = filedialog.askopenfilenames(
        title="Select simulation result(s)",
        filetypes=[("CSV files", "*.csv")]
    )
    root.destroy()

    if not squid_path or not sim_paths:
        print("Selection cancelled.")
        return

    # --- Output directory ---
    ts     = datetime.now().strftime("%Y%m%d_%H%M%S")
    outdir = f"Comparison_Results_{ts}"
    os.makedirs(outdir, exist_ok=True)

    # --- Load averaged SQUID data ---
    df_sq = pd.read_csv(squid_path)

    # Support both old (Field_Oe) and new (Field_T) column names
    if 'Field_T' in df_sq.columns:
        h_sq = df_sq['Field_T'].values
    elif 'Field_Oe' in df_sq.columns:
        h_sq = df_sq['Field_Oe'].values * 1e-4  # convert Oe → T
    else:
        print("[!] Could not find field column in SQUID CSV. Expected 'Field_T' or 'Field_Oe'.")
        return

    # Normalised branches (bottom panel)
    m_sq_norm_desc = df_sq['M_norm_desc'].values
    m_sq_norm_asc  = df_sq['M_norm_asc'].values

    # Physical moment branches (top panel) — converted from emu to A·m²
    # so both SQUID and simulation are on the same axis (1 emu = 1e-3 A·m²)
    m_sq_emu_desc  = df_sq['M_emu_desc'].values * EMU_TO_AM2
    m_sq_emu_asc   = df_sq['M_emu_asc'].values  * EMU_TO_AM2

    # --- Build two-panel figure ---
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 12))
    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

    # SQUID on both panels as solid black closed loops (desc + asc)
    # zorder=10 keeps it visually on top of all simulation lines
    ax1.plot(h_sq, m_sq_emu_desc, color='black', lw=2, label='SQUID (mean)', zorder=10)
    ax1.plot(h_sq, m_sq_emu_asc,  color='black', lw=2, zorder=10)

    ax2.plot(h_sq, m_sq_norm_desc, color='black', lw=2, label='SQUID (mean)', zorder=10)
    ax2.plot(h_sq, m_sq_norm_asc,  color='black', lw=2, zorder=10)

    # --- Load and overlay each simulation file ---
    for i, p in enumerate(sim_paths):
        try:
            df_sim = pd.read_csv(p)

            # Auto-detect field column
            h_col = next(
                (c for c in df_sim.columns if any(k in c for k in SIM_FIELD_KEYWORDS)),
                None
            )

            if h_col is None:
                print(f"[!] Skipping {os.path.basename(p)}: field column not found.")
                continue

            h_sim = df_sim[h_col].values

            # Truncate label for legend readability
            label = os.path.basename(p).replace('.csv', '')[:40]
            color = colors[i % len(colors)]

            # Top panel: physical moment [A·m²]
            if SIM_MAG_PHYS in df_sim.columns:
                ax1.plot(h_sim, df_sim[SIM_MAG_PHYS].values,
                         linestyle='--', lw=1.5, alpha=0.85,
                         color=color, label=f"Sim: {label}")
            else:
                print(f"[!] '{SIM_MAG_PHYS}' column not found in {os.path.basename(p)} — skipping top panel.")

            # Bottom panel: normalised Mz/Ms
            if SIM_MAG_NORM in df_sim.columns:
                ax2.plot(h_sim, df_sim[SIM_MAG_NORM].values,
                         linestyle='--', lw=1.5, alpha=0.85,
                         color=color, label=f"Sim: {label}")
            else:
                print(f"[!] '{SIM_MAG_NORM}' column not found in {os.path.basename(p)} — skipping bottom panel.")

        except Exception as e:
            print(f"[!] Error loading {os.path.basename(p)}: {e}")

    # --- Axes formatting ---
    squid_name = os.path.basename(squid_path)

    ax1.set_ylabel(r"$m$ (A·m²)", fontsize=13)
    ax1.set_title(
        f"SQUID vs. Simulations\nSQUID: {squid_name}",
        fontsize=12, fontweight='bold'
    )

    ax2.set_ylabel(r"$M/M_{\mathrm{sat}}$", fontsize=13)
    ax2.set_xlabel(r"$\mu_0 H$ (T)", fontsize=13)

    for ax in [ax1, ax2]:
        ax.axhline(0, color='black', lw=0.5)
        ax.axvline(0, color='black', lw=0.5)
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.legend(loc='best', fontsize='small', frameon=True)

    fig.tight_layout()

    # --- Save outputs ---
    plot_base = os.path.join(outdir, f"SQUID_Sim_Comparison_{ts}")
    fig.savefig(plot_base + ".png", dpi=300)
    fig.savefig(plot_base + ".svg")
    plt.close(fig)

    print(f"\nDONE. Comparison plot saved in: {outdir}")
    print(f"  → {os.path.basename(plot_base)}.png / .svg")


if __name__ == "__main__":
    main()
